In [9]:
# Extract features for radicals (run once)

import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import numpy as np

class RadicalDataset(Dataset):
    def __init__(self, folder):
        self.paths = [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".png")]
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=3),
            transforms.Resize(224),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = self.paths[idx]
        img = Image.open(img_path).convert("L")
        img = self.transform(img)
        return img, os.path.basename(img_path)

class DINOFeatureExtractor(torch.nn.Module):
    def __init__(self, model_path):
        super().__init__()
        from torchvision import models
        resnet = models.resnet18(pretrained=False)
        self.backbone = torch.nn.Sequential(*list(resnet.children())[:-1])
        self.projector = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(512, 2048),
            torch.nn.GELU(),
            torch.nn.Linear(2048, 2048),
            torch.nn.GELU(),
            torch.nn.Linear(2048, 512),
        )
        self.load_state_dict(torch.load(model_path))
        self.eval()

    def forward(self, x):
        x = self.backbone(x)
        x = self.projector(x)
        return x

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_path = "/app/notebooks/dino_radicals_resnet.pth"
    radical_folder = "/app/data/radicals"
    output_folder = "/app/notebooks/radical_features"
    os.makedirs(output_folder, exist_ok=True)

    dataset = RadicalDataset(radical_folder)
    loader = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=0)

    model = DINOFeatureExtractor(model_path).to(device)

    with torch.no_grad():
        for imgs, names in tqdm(loader, desc="Extracting Radical Features"):
            imgs = imgs.to(device)
            feats = model(imgs).cpu().numpy()
            for f, name in zip(feats, names):
                np.save(os.path.join(output_folder, name.replace(".png", ".npy")), f)

    print(f"✅ Radical features saved in {output_folder}")


/usr/local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Extracting Radical Features: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 31/31 [00:04<00:00,  6.54it/s]

✅ Radical features saved in /app/notebooks/radical_features
